In [ ]:
%pip install requests

In [7]:
import os
import json
import requests
import weaviate

from dotenv import load_dotenv
from weaviate.auth import AuthApiKey
from weaviate.classes.config import Configure, Property, DataType, VectorDistances
from weaviate.classes.query import MetadataQuery

load_dotenv()

WEAVIATE_URL = os.getenv("WEAVIATE_REST_ENDPOINT")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY_ADMIN")

if not WEAVIATE_URL or not WEAVIATE_API_KEY:
    raise ValueError("Set WEAVIATE_URL and WEAVIATE_API_KEY in your .env file.")

def json_print(data):
    # default=str handles values such as UUIDs and datetime objects.
    print(json.dumps(data, indent=4, ensure_ascii=False, default=str))

In [8]:
# Optional provider key for external vectorizers such as text2vec_openai.
# Not needed when using text2vec_weaviate.
provider_headers = {}
if os.getenv("OPENAI_API_KEY"):
    provider_headers["X-OpenAI-Api-Key"] = os.environ["OPENAI_API_KEY"]

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=AuthApiKey(WEAVIATE_API_KEY),
    headers=provider_headers or None,
)

print("Connected:", client.is_connected())
print("Ready:", client.is_ready())

Connected: True
Ready: True


d:\epam\sadhaka\module_3\rag\vectordb\.venv\Lib\site-packages\weaviate\warnings.py:355: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(


In [9]:
json_print(client.get_meta())

{
    "grpcMaxMessageSize": 104858000,
    "hostname": "http://[::]:8080",
    "license": {
        "documentationHref": "https://docs.weaviate.io/deploy/enterprise",
        "edition": "enterprise",
        "licenseId": "lic_06GH1AHSN9X8FG04Y3N3AG5P2W",
        "status": "valid"
    },
    "modules": {
        "backup-s3": {
            "bucketName": "prod-wcs-data-plane-eu-central-1-workloads-backups",
            "endpoint": "s3.amazonaws.com",
            "rootName": "3c63b57c-c797-4995-8a6f-c13e16463d64",
            "useSSL": true
        },
        "generative-anthropic": {
            "documentationHref": "https://docs.anthropic.com/en/api/getting-started",
            "name": "Generative Search - Anthropic"
        },
        "generative-anyscale": {
            "documentationHref": "https://docs.anyscale.com/endpoints/overview",
            "name": "Generative Search - Anyscale"
        },
        "generative-aws": {
            "documentationHref": "https://docs.aws.amazon.c

In [10]:
DATA_URL = (
    "https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/"
    "data/jeopardy_tiny.json"
)

response = requests.get(DATA_URL, timeout=30)
response.raise_for_status()
data = response.json()

print(f"Loaded {len(data)} records")
json_print(data[0])

Loaded 10 records
{
    "Category": "SCIENCE",
    "Question": "This organ removes excess glucose from the blood & stores it as glycogen",
    "Answer": "Liver"
}


In [12]:
COLLECTION_NAME = "Questions"

if(not client.collections.exists(COLLECTION_NAME)):
    client.collections.create(
        name=COLLECTION_NAME,
        vector_config=Configure.Vectors.text2vec_weaviate(
            vector_index_config=Configure.VectorIndex.hfresh(
                distance_metric=VectorDistances.COSINE
            )
        ),
        properties=[
            Property(name="answer", data_type=DataType.TEXT),
            Property(name="question", data_type=DataType.TEXT),
            Property(name="category", data_type=DataType.TEXT)
        ]
    )

questions = client.collections.get(COLLECTION_NAME)
print("Collection ready: ", COLLECTION_NAME)

Collection ready:  Questions


In [15]:
with questions.batch.dynamic() as batch:
    for row in data:
        batch.add_object(
            properties={
                'question':row["Question"],
                'answer':row['Answer'],
                'category':row['Category']
            }
        )

    if batch.number_errors:
        print("Batch errors: ", batch.number_errors)

failed_objects = questions.batch.failed_objects
print("failed object count: ", len(failed_objects))

if failed_objects:
    json_print(failed_objects[0])

failed object count:  0


In [ ]:
aggregate_result = questions.aggregate.over_all(total_count=True)
print("total objects: ", aggregate_result.total_count)

total objects:  10


In [17]:
sample_result = questions.query.fetch_objects(limit=1)
if sample_result.objects:
    sample_object = sample_result.objects[0]
    print("UUID: ", sample_object.uuid)
    json_print(sample_object.properties)

UUID:  2b4ab161-e8ad-4f0b-859e-c4bf8786cfac
{
    "answer": "the nose or snout",
    "question": "The gavial looks very much like a crocodile except for this bodily feature",
    "category": "ANIMALS"
}


In [30]:
result = questions.query.fetch_object_by_id(
    uuid=sample_object.uuid,
    include_vector=True
)

print("UUID: ", result.uuid)
json_print(result.properties)
vector = result.vector["default"]
print(next(iter(result.vector.values())))



UUID:  2b4ab161-e8ad-4f0b-859e-c4bf8786cfac
{
    "answer": "the nose or snout",
    "question": "The gavial looks very much like a crocodile except for this bodily feature",
    "category": "ANIMALS"
}
[-0.045691851526498795, -0.020639216527342796, 0.033045001327991486, -0.09160622954368591, 0.01057922188192606, -0.015066813677549362, -0.02536787837743759, 0.022734662517905235, -0.06141696870326996, 0.015270795673131943, -0.0045988550409674644, -0.025423510000109673, -0.035492777824401855, 0.03456559032201767, -0.05726316571235657, 0.016021817922592163, -0.01921134628355503, 0.011135534383356571, -0.0383114330470562, -0.03541860356926918, -0.05092119425535202, 0.023272432386875153, -0.049845658242702484, 0.011191166006028652, -0.02036106027662754, -0.017829835414886475, -0.04012871906161308, 0.020898830145597458, 0.029874015599489212, 0.08196347206830978, 0.04658195376396179, 0.006596946623176336, -0.030597222968935966, -0.0012945870403200388, 0.06212163344025612, -0.00846523139625787